Seemingly somewhere in the emrging of the csvs in the indicator script, for some coutnries some months were dropped, therefore from the single dfs, they will be merged new into set final df

In [1]:
import pandas as pd

In [29]:
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)

In [2]:
def pivot_table(df, index, columns, values):
    pivot = df.pivot_table(
        index=index,
        columns=columns,
        values=values,
        aggfunc='sum'
    ).reset_index().fillna(0)
    return pivot

def build_pivot_set(df, index_col, group_col, value_cols, rename_fn):
    tables = [
        rename_fn(pivot_table(df, index_col, group_col, col), col)
        for col in value_cols
    ]
    result = tables[0]
    for t in tables[1:]:
        result = result.merge(t, on=index_col)
    return result

In [3]:
def rename_contrib_type(df, contrib_type):
    df = df.rename(columns={
        'CREATION':f'{contrib_type}_creation',
         'DELETION':f'{contrib_type}_deletion',
         'GEOMETRY':f'{contrib_type}_geometry',
        'TAG':f'{contrib_type}_tag',
        'TAG_GEOMETRY':f'{contrib_type}_tag_geometry'
    })
    return df

def rename_feature_type(df, feature_type):
    df = df.rename(columns={
        'building':f'{feature_type}_building',
         'highway':f'{feature_type}_highway',
         'other':f'{feature_type}_other'
    })
    return df

In [11]:
months = pd.read_csv("months.csv")

In [5]:
contrib_cols = ['contributors', 'contributors_ai', 'changesets', 'changesets_ai', 'edits', 'edits_ai']
feature_specs = ['contributors', 'contributors_ai', 'share_contributors', 'edits','edits_ai', 'share_edits', 'changesets', 'changesets_ai', 'share_changesets']

In [50]:
def update_indicator_df(country):
    monthly_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_monthly_stats.csv")
    contrib_type_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_contrib_type.csv")
    feature_type_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_feature_type.csv")
    xp_monthly = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_updated_monthly_experience_ohsome.csv")
    affiliation_all_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_affiliation_all.csv")
    affiliation_hot_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_affiliation_hot.csv")
    affiliation_ai_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_affiliation_hot.csv")
    retention_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_retention.csv")
    survival_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_survival.csv")
    monthly_interaction_network_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_monthly_interaction_network.csv")
    comments_notes_df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_comments_notes_total.csv")

    contrib_type_df_long = build_pivot_set(contrib_type_df, 'months', 'contrib_type', contrib_cols, rename_contrib_type)
    feature_type_df_long = build_pivot_set(feature_type_df, 'months', 'tag_type', feature_specs, rename_feature_type)

    #creating one big csv with all monthly indicators
    #using the complete months as first input, so all needed months are there, useing left join for all others so all months are guaranteed
    big_monthly_df = months.merge(monthly_df[["months", "editsPerc", "changesetsPerc", "contributorsPerc", "global_new_contrib", "local_new_contrib", "new_contrib_ai", "new_contrib_non_ai"]], on ="months", how="left")

    big_monthly_df = big_monthly_df.merge(affiliation_all_df, on ="months", how="left")
    big_monthly_df = big_monthly_df.merge(xp_monthly, on = "months", how="left")
    big_monthly_df = big_monthly_df.merge(comments_notes_df, on = "months", how="left")
    big_monthly_df = big_monthly_df.merge(contrib_type_df_long, on = "months", how="left")
    big_monthly_df = big_monthly_df.merge(feature_type_df_long, on = "months", how="left")
    big_monthly_df = big_monthly_df.merge(retention_df, on = "months", how="left")
    big_monthly_df = big_monthly_df.merge(monthly_interaction_network_df[["months", "mean_degree"]], on = "months", how="left")
    
    #calculating massing ratios / shares
    big_monthly_df["changesets_corporate_share"] =big_monthly_df["changesets_corporate"] / big_monthly_df["changesets"]
    big_monthly_df["contributors_corporate_share"] =big_monthly_df["contributors_corporate"] / big_monthly_df["contributors"]
    
    big_monthly_df["changesets_humanitarian_share"] =big_monthly_df["changesets_humanitarian"] / big_monthly_df["changesets"]
    big_monthly_df["contributors_humanitarian_share"] =big_monthly_df["contributors_humanitarian"] / big_monthly_df["contributors"]
    
    big_monthly_df["changesets_ai_share"]= big_monthly_df["changesetsPerc"]/100
    big_monthly_df["contributors_ai_share"]= big_monthly_df["contributorsPerc"]/100
    big_monthly_df["new_contributors_ai_share"]= big_monthly_df["new_contrib_ai"] / (big_monthly_df["global_new_contrib"] + big_monthly_df["local_new_contrib"])
    
    big_monthly_df["contributors_prolific_share_ai_total"]= big_monthly_df["contributors_prolific_ai"]/big_monthly_df["contributors"]
    big_monthly_df["contributors_casual_share_ai_total"]= big_monthly_df["contributors_casual_ai"]/big_monthly_df["contributors"]
    big_monthly_df["contributors_prolific_share_total"]= big_monthly_df["contributors_prolific_total"]/big_monthly_df["contributors"]
    big_monthly_df["contributors_casual_share_total"]= big_monthly_df["contributors_casual_total"]/big_monthly_df["contributors"]
    
    big_monthly_df["contributors_inactive_share_total"]= big_monthly_df["contributors_inactive_total"]/big_monthly_df["contributors"]
    big_monthly_df["contributors_inactive_share_ai_total"]= big_monthly_df["contributors_inactive_ai"]/big_monthly_df["contributors"]
    
    big_monthly_df["global_new_contrib_share"]= big_monthly_df["global_new_contrib"]/big_monthly_df["contributors"]
    big_monthly_df["building_changesets_share"] = big_monthly_df["changesets_building"] / big_monthly_df["changesets"]
    big_monthly_df["highway_changesets_share"] = big_monthly_df["changesets_highway"] / big_monthly_df["changesets"]
    big_monthly_df["building_changesets_ai_share"] = big_monthly_df["changesets_ai_building"] / big_monthly_df["changesets"]
    big_monthly_df["highway_changesets_ai_share"] = big_monthly_df["changesets_ai_highway"] / big_monthly_df["changesets"]
    
    #added from Spearman 
    big_monthly_df["total_new_contrib"] = big_monthly_df["global_new_contrib"] + big_monthly_df["local_new_contrib"]
    big_monthly_df["edits_corporate_share"] =big_monthly_df["edits_corporate"] / big_monthly_df["edits"]
    big_monthly_df["edits_humanitarian_share"] =big_monthly_df["edits_humanitarian"] / big_monthly_df["edits"]
    big_monthly_df["edits_ai_share"]= big_monthly_df["editsPerc"]/100
    big_monthly_df["edits_building_share"]= big_monthly_df["share_edits_building"]/100
    big_monthly_df["edits_other_share"]= big_monthly_df["share_edits_other"]/100
    big_monthly_df["edits_highway_share"]= big_monthly_df["share_edits_highway"]/100
    
    #new columns --> need new measure for contributors_AI since there are lsight differences between duckdb queries and ohsome_now query
    big_monthly_df["contributors_ai_ohsome"] = big_monthly_df["contributors_inactive_ai"] + big_monthly_df["contributors_prolific_ai"] +big_monthly_df["contributors_casual_ai"]
    big_monthly_df["contributors_inactive_share_ai"]= big_monthly_df["contributors_inactive_ai"]/big_monthly_df["contributors_ai_ohsome"]
    big_monthly_df["contributors_prolific_share_ai"]= big_monthly_df["contributors_prolific_ai"]/big_monthly_df["contributors_ai_ohsome"]
    big_monthly_df["contributors_casual_share_ai"]= big_monthly_df["contributors_casual_ai"]/big_monthly_df["contributors_ai_ohsome"]
    
    big_monthly_df['new_contrib_ai_share'] = big_monthly_df['new_contrib_ai']/big_monthly_df['total_new_contrib']

    big_monthly_df = big_monthly_df.fillna(0)
    indicators_df = big_monthly_df[['months', 'contributors', 'changesets', 'edits','mean_degree', 'active_users', 'total_new_contrib' , 
    'changeset_comments_ai', 'ratio_active_users_ai',
    'changesets_corporate_share', 'contributors_corporate_share', 'changesets_humanitarian_share', 'contributors_humanitarian_share', 
    'edits_corporate_share', 'edits_humanitarian_share', 'edits_ai_share', 
    'changesets_ai_share', 'contributors_ai_share', 'contributors_prolific_share_ai', 'contributors_casual_share_ai','contributors_inactive_share_ai',
    'building_changesets_share', 'highway_changesets_share', 'building_changesets_ai_share', 'highway_changesets_ai_share',
    'new_contributors_ai_share', 'notes_length',
    'contributors_prolific_share_ai_total', 'contributors_casual_share_ai_total', 'contributors_inactive_share_ai_total','retention_rate_ai', 'new_contrib_ai_share'
    ]].copy()

    indicators_df.to_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_prepped_indicators_stat_update.csv", index = False)
    big_monthly_df.to_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_all_monthly_metrics_update.csv", index = False)


In [51]:
update_indicator_df("ALB")

In [56]:
countries = ["AFG", "ALB", "BGD", "BOL", "BTN", "BRN", "COL", "CYP", "ECU", "ETH", "B35", "GRC", "HUN", "IND", "KEN", "KHM", "KOR",
             "LAO", "MAR", "MKD", "MNE", "NGA", "NZ1", "PNG", "SAU", "SLE", "TUR", "TZA", "URY", "US1", "VNM"
            ]


In [57]:
for c in countries:
    update_indicator_df(c)

In [52]:
alb_monthly_df = pd.read_csv(r"..\..\data\ouput\country_analysis\v5_indicators\ALB\ALB_all_monthly_metrics_update.csv")


In [53]:
alb_indicators_df =pd.read_csv(r"..\..\data\ouput\country_analysis\v5_indicators\ALB\ALB_prepped_indicators_stat_update.csv")


In [59]:
alb_monthly_df[["changeset_comments_nonai"]]

,changeset_comments_nonai
0,0.232587
1,1.862642
2,0.118172
3,0.008256
4,0.161773
5,10.106802
6,0.002749
7,2.210939
8,0.000000
9,6.187205


In [55]:
alb_monthly_df[["months", "contributors_prolific_share_ai","contributors_prolific_ai", "contributors_casual_ai", "contributors_inactive_ai", "contributors_AI"]]

,months,contributors_prolific_share_ai,contributors_prolific_ai,contributors_casual_ai,contributors_inactive_ai,contributors_AI
0,2020-01,0.000000,0.0,0.0,0.0,2
1,2020-02,0.000000,0.0,0.0,0.0,2
2,2020-03,0.000000,0.0,0.0,0.0,0
3,2020-04,0.000000,0.0,1.0,0.0,2
4,2020-05,1.000000,1.0,0.0,0.0,1
5,2020-06,0.000000,0.0,0.0,0.0,0
6,2020-07,0.000000,0.0,0.0,0.0,0
7,2020-08,1.000000,1.0,0.0,0.0,0
8,2020-09,0.500000,1.0,1.0,0.0,2
9,2020-10,0.500000,1.0,1.0,0.0,1


In [34]:
import numpy as np

In [36]:
print(alb_monthly_df[['contributors_prolific_share_ai_total']].dtypes)
print(alb_indicators_df[['contributors_prolific_share_ai_total']].dtypes)

contributors_prolific_share_ai_total    float64
dtype: object
contributors_prolific_share_ai_total    float64
dtype: object


In [35]:
print(np.isinf(alb_monthly_df['contributors_prolific_share_ai_total']).sum())
print(np.isinf(alb_indicators_df['contributors_prolific_share_ai_total']).sum())

0
0
